In [2]:
import ee
import pandas as pd

In [3]:
ee.Initialize(project="sundarbans-climate-analysis")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [4]:
print("Google Earth Engine initialized successfully!")

Google Earth Engine initialized successfully!


In [5]:
districts = ee.FeatureCollection(
    "FAO/GAUL/2015/level2"
)

C:\Users\Projit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level2

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level2

  warnings.warn(warning, category=DeprecationWarning)


In [6]:
bangladesh_districts = districts.filter(
    ee.Filter.eq(
        "ADM0_NAME",
        "Bangladesh"
    )
)

In [7]:
target_districts = bangladesh_districts.filter(
    ee.Filter.inList(
        "ADM2_NAME",
        [
            "Satkhira",
            "Khulna",
            "Bagerhat"
        ]
    )
)

In [8]:
print(
    "Selected districts:"
)

Selected districts:


In [9]:
print(
    target_districts
    .aggregate_array("ADM2_NAME")
    .getInfo()
)

['Bagerhat', 'Khulna', 'Satkhira']


In [10]:
study_area = target_districts.geometry()

print(
    "Study area created successfully."
)

Study area created successfully.


In [11]:
era5 = ee.ImageCollection(
    "ECMWF/ERA5_LAND/MONTHLY_AGGR"
)

In [13]:
rainfall_collection = (
    era5
    .filterDate(
        "2000-01-01",
        "2026-01-01"
    )
    .select(
        "total_precipitation_sum"
    )
)

In [14]:
print(
    "Number of monthly images:",
    rainfall_collection.size().getInfo()
)

Number of monthly images: 312


In [15]:
start_date = ee.Date(
    "2000-01-01"
)

In [16]:
end_date = ee.Date(
    "2026-01-01"
)

In [17]:
number_of_months = (
    end_date
    .difference(
        start_date,
        "month"
    )
)

In [18]:
month_sequence = ee.List.sequence(
    0,
    number_of_months.subtract(1)
)

In [22]:
def calculate_monthly_rainfall(month_number):

    # Current month
    current_date = start_date.advance(
        month_number,
        "month"
    )

    # Next month
    next_date = current_date.advance(
        1,
        "month"
    )


    # Get image for the current month

    monthly_image = (
        rainfall_collection
        .filterDate(
            current_date,
            next_date
        )
        .first()
    )

    rainfall_meters = (
        monthly_image
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=11132,
            maxPixels=1000000
        )
        .get(
            "total_precipitation_sum"
        )
    )

    rainfall_mm = (
        ee.Number(rainfall_meters)
        .multiply(1000)
    )


    # Create feature

    return ee.Feature(
        None,
        {
            "Year":
                current_date.get("year"),

            "Month_Number":
                current_date.get("month"),

            "Month":
                current_date.format("MMMM"),

            "Date":
                current_date.format(
                    "YYYY-MM-dd"
                ),

            "Rainfall_mm":
                rainfall_mm
        }
    )


In [23]:
monthly_rainfall = (
    ee.FeatureCollection(
        month_sequence.map(
            calculate_monthly_rainfall
        )
    )
)

In [24]:
print(
    "Monthly rainfall calculation completed."
)


Monthly rainfall calculation completed.


In [25]:
print(
    "Number of records:",
    monthly_rainfall.size().getInfo()
)

Number of records: 312


In [26]:
data = monthly_rainfall.getInfo()

rows = []

for feature in data["features"]:

    properties = feature["properties"]

    rows.append(
        {
            "Year":
                properties.get("Year"),

            "Month_Number":
                properties.get(
                    "Month_Number"
                ),

            "Month":
                properties.get("Month"),

            "Date":
                properties.get("Date"),

            "Rainfall_mm":
                properties.get(
                    "Rainfall_mm"
                )
        }
    )


df_rainfall = pd.DataFrame(
    rows
)

In [27]:
df_rainfall = (
    df_rainfall
    .sort_values(
        by="Date"
    )
    .reset_index(
        drop=True
    )
)

In [28]:
df_rainfall[
    "Rainfall_mm"
] = (
    df_rainfall[
        "Rainfall_mm"
    ]
    .round(2)
)

In [29]:
print(
    df_rainfall.head(12)
)

print(
    "\nTotal records:",
    len(df_rainfall)
)

    Year  Month_Number      Month        Date  Rainfall_mm
0   2000             1    January  2000-01-01         7.85
1   2000             2   February  2000-02-01        24.16
2   2000             3      March  2000-03-01        11.18
3   2000             4      April  2000-04-01        59.38
4   2000             5        May  2000-05-01       198.05
5   2000             6       June  2000-06-01       334.97
6   2000             7       July  2000-07-01       339.04
7   2000             8     August  2000-08-01       285.75
8   2000             9  September  2000-09-01       252.15
9   2000            10    October  2000-10-01       179.99
10  2000            11   November  2000-11-01         6.13
11  2000            12   December  2000-12-01         0.03

Total records: 312


In [30]:
output_file = (
    "Sundarbans_Monthly_Rainfall_2000_2025.xlsx"
)

df_rainfall.to_excel(
    output_file,
    index=False
)


print(
    "\nExcel file created successfully:"
)

print(
    output_file
)


Excel file created successfully:
Sundarbans_Monthly_Rainfall_2000_2025.xlsx


In [32]:
print(
    "\nDate range:"
)

print(
    df_rainfall["Date"].min(),
    "to",
    df_rainfall["Date"].max()
)


print(
    "\nMissing values:"
)

print(
    df_rainfall.isnull().sum()
)


print(
    "\nLast 12 months:"
)

print(
    df_rainfall.tail(12)
)


Date range:
2000-01-01 to 2025-12-01

Missing values:
Year            0
Month_Number    0
Month           0
Date            0
Rainfall_mm     0
dtype: int64

Last 12 months:
     Year  Month_Number      Month        Date  Rainfall_mm
300  2025             1    January  2025-01-01         1.61
301  2025             2   February  2025-02-01        12.29
302  2025             3      March  2025-03-01         6.16
303  2025             4      April  2025-04-01       104.12
304  2025             5        May  2025-05-01       209.89
305  2025             6       June  2025-06-01       387.29
306  2025             7       July  2025-07-01       614.95
307  2025             8     August  2025-08-01       323.58
308  2025             9  September  2025-09-01       265.17
309  2025            10    October  2025-10-01       148.95
310  2025            11   November  2025-11-01         5.48
311  2025            12   December  2025-12-01         0.25
